---
# <div style="text-align: center"> Introduction </div>
---

Along these tutorials, we will see how <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code interact to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System**, **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**
6) How to Run <span style="color:blue">**SCOPE**</span> - Part 1: The Environment class
7) How to Run <span style="color:blue">**SCOPE**</span> - Part 2: Simple Setup
8) How to Run <span style="color:blue">**SCOPE**</span> - Part 3: Detailed Actions

---
# <div style="text-align: center"> Tutorial 6: How to Run SCOPE, Part 1</div>
---

## File Structure

SCOPE interacts with five types of files:   

- Binary files containing SYSTEM-class objects, discussed in **Tutorial 1** (<span style="color:green">**SYSTEM**</span>)
- Plain text (or binary) files that are used to create SOURCES for systems, as discussed in **Tutorial 1** (<span style="color:navy">**SOURCE**</span>)
- Plain text input, output and submission files associated with the computation (<span style="color:brown">**COMPUTATIONS**</span>)  
- Plain text SCOPE input FILES describing tasks, discussed in **Tutorial 5** (<span style="color:orange">**INPUT**</span>)
- Binary files containing ENVIRONMENT-class objects, as we will discuss here (<span style="color:purple">**ENVIRONMENT**</span>)   

The first three accumulate very rapidly, so they are stored in SYSTEM-specific folders, whereas <span style="color:orange">**INPUT**</span> and <span style="color:purple">**ENVIRONMENT**</span> files are specific of each project 


<div style="text-align: center;">
  <figure style="display: inline-block; text-align: center; width: 400px; margin: 0;">
    <img src="../images/scheme1.png" alt="scheme1" width="400"/>
    <figcaption style="font-size: 0.9em; line-height: 1.4; text-align: justify; margin-top: 6px;">
      <b>Scheme 1</b>: Intended folder structure for projects in SCOPE. Environment and Input files inside the project folder (in black). Systems (green), Sources (blue), and Computations (red) folders have dedicated subfolders for each system.
    </figcaption>
  </figure>
</div>

SCOPE minimizes the interaction between the user and the projects files, thanks to the <span style="color:purple">**ENVIRONMENT**</span> Class, which stores all relevant paths in a given computer. This also simplifies sharing data between the local computer (which is where you would typically analyse the results) and the computational clusters (where you run the computations).

The Key variables in the Environment-class object are: 
- sources_path
- systems_path
- computations_path

Let's say you have a local computer ("LOCAL") and two computational HPC clusters ("HPC1" and "HPC2") at your disposal. To move files between these computers, you just need to:
- Create an environment in each computer. That is, run "scope_config" in LOCAL, HPC1 and HPC2
- Run you SCOPE tasks in HPC1 and/or HPC2
- Syncronize the LOCAL folders with the files in HPC1 and HPC2
- In principle, whenever SCOPE needs to know the actual path of a source, system, or computation file, it will take it from the evironment.

## Dos and don'ts

#### - **Project**: 

- You can freely move the main folder of the project (black folder in scheme 1), as long as you update all environment paths, running env.set_paths() on the existing environment, or creating a new one

#### - **Systems**: 

- Can be deleted or restarted harmlessly as long as COMPUTATION files remain in place (i.e. same path, same filename).   
- You can move the entire Systems folder, preserving the file structure inside, and updating the env.systems_path attribute of the existing environment
- Ideally, update all systems (one by one) with the SYSTEM-class function: sys.set_paths_from_environment(env), where env and sys are the environment and system objects 

#### - **Sources**: 

- Once imported to the SYSTEM, individual files can be deleted, or even the whole folder if you wish

#### - **Computation files**:
- <span style="color:red">Do not</span> remove those files unless you're really sure what you're doing, even if the results have been registered already
- If you move the Computations folder, preserve the file structure inside, and change the env.computations_path in the environment  
  Otherwise, SCOPE will re-submit these very same computations, wasting computational resources (!)

#### - **Job Files**: 

- You can move or delete these files harmlessly
- You can modify the *&environment* and *&options* sections anytime during the execution of tasks (Cases 1-4 below)
- <span style="color:red">AVOID</span> modifying the *&job_data* and *&qc_data* sections as much as possible. 

---
## Example:
---

Imagine you want to run 2 tasks (*task1*, *task2*), and that *task2* should only start when *task1* finishes. You can specify all this in the input files  
Then, you can run **SCOPE** with the following command:
```bash
scope_run_task -n ../scope_env.npy -s ABITEM/ABITEM.npy -j task1.task task2.task
````

In [ ]:
## The example below serves to illustrate how SCOPE must be executed. Imagine you want to run 2 jobs (job1, job2), 
## in which job2 can only start when job1 finishes. You can do so with the following command:

## scope_run_job -n ../scope_env.npy -s ABITEM/ABITEM.npy -j job1.job job2.job

## Case 1) The first time you execute this command, if everything goes well, Scope will submit the computation(s) associated to job1 to the cluster, and won't do anything else. The System file will be updated
## Case 2) If you execute once again while job1's computations are still running, nothing will happen. The System file won't be updated
## Case 3) Once job1's computations finish, if you execute, Scope will register job1, and submit job2. The System file will be updated
## Case 4) When job2's computations finish, if you execute, Scope will skip job1 (it is already registered), register job2, and the workflow will be over. The System file will be updated, and you can then analyse the results. 

# Example

In [ ]:
import os
import scope
from scope.read_write import *